In [1]:
%pip install ucimlrepo

In [2]:
# Import Libraries
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from IPython.display import display
from typing import Any, Dict

"""
Load the Cardiotocography dataset from UCI repository.
Reference:
    https://archive.ics.uci.edu/dataset/193/cardiotocography
"""

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

CTG_TARGET_COLS = ["NSP", "CLASS"]
NSP_LABELS = {1: "Normal", 2: "Suspect", 3: "Pathologic"}

# 1.Data Loading
def load_cardiotocography_dataset() -> pd.DataFrame:

    ctg = fetch_ucirepo(id=193)

    if ctg.data is None:
        raise ValueError("Failed to fetch dataset from UCI repository.")

    # Combine features and targets
    df = pd.concat([ctg.data.features, ctg.data.targets], axis=1)

    return df

# 1.1 Exploring dataset
def explore_dataset(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Returns:
        dict: Summary statistics including column types, shape, etc.
    """
    # Identify column types
    cat_cols = [col for col in df.columns if df[col].dtype == 'object']
    num_cols = [col for col in df.columns if df[col].dtype != 'object']

    summary = {
        'shape': df.shape,
        'categorical_columns': cat_cols,
        'numerical_columns': num_cols,
        'total_samples': df.shape[0],
        'total_features': df.shape[1]
    }

    # Display overview
    print("1. DATASET OVERVIEW")
    display(df.head(10))
    print(f"\nColumn Data Types:")
    print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
    print(f"Numerical columns ({len(num_cols)}): {num_cols}")
    print(f"\nDataset Summary:")
    print(f"Total samples: {summary['total_samples']:,}")
    print(f"Total features: {summary['total_features']}")

    return summary
# Load and explore data
df = load_cardiotocography_dataset()
dataset_summary = explore_dataset(df)

1. DATASET OVERVIEW


,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,MLTV,Width,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,CLASS,NSP
0,120,0.000,0.0,0.000,0.000,0.0,0.000,73,0.5,43,2.4,64,62,126,2,0,120,137,121,73,1,9,2
1,132,0.006,0.0,0.006,0.003,0.0,0.000,17,2.1,0,10.4,130,68,198,6,1,141,136,140,12,0,6,1
2,133,0.003,0.0,0.008,0.003,0.0,0.000,16,2.1,0,13.4,130,68,198,5,1,141,135,138,13,0,6,1
3,134,0.003,0.0,0.008,0.003,0.0,0.000,16,2.4,0,23.0,117,53,170,11,0,137,134,137,13,1,6,1
4,132,0.007,0.0,0.008,0.000,0.0,0.000,16,2.4,0,19.9,117,53,170,9,0,137,136,138,11,1,2,1
5,134,0.001,0.0,0.010,0.009,0.0,0.002,26,5.9,0,0.0,150,50,200,5,3,76,107,107,170,0,8,3
6,134,0.001,0.0,0.013,0.008,0.0,0.003,29,6.3,0,0.0,150,50,200,6,3,71,107,106,215,0,8,3
7,122,0.000,0.0,0.000,0.000,0.0,0.000,83,0.5,6,15.6,68,62,130,0,0,122,122,123,3,1,9,3
8,122,0.000,0.0,0.002,0.000,0.0,0.000,84,0.5,5,13.6,68,62,130,0,0,122,122,123,3,1,9,3
9,122,0.000,0.0,0.003,0.000,0.0,0.000,86,0.3,6,10.6,68,62,130,1,0,122,122,123,1,1,9,3



Column Data Types:
Categorical columns (0): []
Numerical columns (23): ['LB', 'AC', 'FM', 'UC', 'DL', 'DS', 'DP', 'ASTV', 'MSTV', 'ALTV', 'MLTV', 'Width', 'Min', 'Max', 'Nmax', 'Nzeros', 'Mode', 'Mean', 'Median', 'Variance', 'Tendency', 'CLASS', 'NSP']

Dataset Summary:
Total samples: 2,126
Total features: 23


In [3]:
# 4 Bootstrapping
"""
Using SMOTE in order to generate synthetic minority samples
"""
from imblearn.over_sampling import SMOTE

X = df.drop(columns=['NSP'])
y = df['NSP']

smote = SMOTE(random_state = 42)
X_resampled, y_resampled = smote.fit_resample(X, y)

smote_df = pd.concat([X_resampled, y_resampled], axis = 1)
smote_df.head()

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,MLTV,Width,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,CLASS,NSP
0,120,0.000,0.0,0.000,0.000,0.0,0.0,73,0.5,43,2.4,64,62,126,2,0,120,137,121,73,1,9,2
1,132,0.006,0.0,0.006,0.003,0.0,0.0,17,2.1,0,10.4,130,68,198,6,1,141,136,140,12,0,6,1
2,133,0.003,0.0,0.008,0.003,0.0,0.0,16,2.1,0,13.4,130,68,198,5,1,141,135,138,13,0,6,1
3,134,0.003,0.0,0.008,0.003,0.0,0.0,16,2.4,0,23.0,117,53,170,11,0,137,134,137,13,1,6,1
4,132,0.007,0.0,0.008,0.000,0.0,0.0,16,2.4,0,19.9,117,53,170,9,0,137,136,138,11,1,2,1


In [5]:
# 5 Split training and test data and create pipeline
"""
Split the data into training and testing sets
and prepare for model testing.
"""

target = 'NSP'

# separate target column
X = smote_df.drop(columns=[target, 'CLASS'])
y = smote_df[target].astype(int)-1

from sklearn.model_selection import train_test_split

# create training and test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.3, random_state = 42, stratify = y
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score, recall_score, precision_score
from sklearn.feature_selection import SelectFromModel

In [6]:
# 5.1 SVM Model Testing
from sklearn import svm
from sklearn.feature_selection import SelectKBest, f_classif

# create SVM model
linear_model = svm.SVC(kernel = 'linear', class_weight='balanced')

# use pipeline in order to scale columns
pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("feature_selection", SelectKBest(f_classif)),
    ("model", linear_model)
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

# calculate evaluation metrics
svm_f1 = f1_score(y_test, y_pred, average='macro')
svm_recall = recall_score(y_test, y_pred, average='macro')
svm_precision = precision_score(y_test, y_pred, average='macro')

In [7]:
# 5.2 Random Forest Testing
from sklearn.ensemble import RandomForestClassifier
# create Random Forest model
random_forest = RandomForestClassifier(class_weight='balanced')

pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("feature_selection", SelectFromModel(RandomForestClassifier(class_weight='balanced'))),
    ("model", random_forest)
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

# calculate evaluation metrics
rf_f1 = f1_score(y_test, y_pred, average='macro')
rf_recall = recall_score(y_test, y_pred, average='macro')
rf_precision = precision_score(y_test, y_pred, average='macro')

In [8]:
# 5.3 XGBoost Model Testing
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# create XGBoost model
xgb_model = xgb.XGBClassifier()
xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2]
}

# use GridSearch to find best hyperparameters
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=xgb_param_grid,
    cv = 5,
    scoring='f1_macro'
)

pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ('selector', SelectFromModel(XGBClassifier(random_state=42))),
    ("model", grid_search)
])

pipe.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)

Best Parameters: {'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 200}


In [9]:
# create new XGBoost model with best parameters
xgb_fitted_model = xgb.XGBClassifier(
    max_depth = 5,
    learning_rate = 0.2,
    n_estimators = 200,
    random_state = 42
)

# fit test and training data to new model
xgb_fitted_model.fit(X_train, y_train)

# test new model
y_fit_pred = xgb_fitted_model.predict(X_test)

# calculate XGBoost evaluation metrics
xgb_f1 = f1_score(y_test, y_fit_pred, average='macro')
xgb_recall = recall_score(y_test, y_fit_pred, average='macro')
xgb_precision = precision_score(y_test, y_fit_pred, average = 'macro')

In [10]:
# create DataFrame of evaluation metrics
scores = pd.DataFrame({"Model": ["Linear SVM", "Random Forest", "XGBoost"],
                      "F1 Score": [svm_f1, rf_f1, xgb_f1],
                      "Recall Score": [svm_recall, rf_recall, xgb_recall],
                      "Precision Score": [svm_precision, rf_precision, xgb_precision]})
# sort by f1 scores
scores = scores.sort_values(by = 'F1 Score', ascending = False)
scores

,Model,F1 Score,Recall Score,Precision Score
2,XGBoost,0.981191,0.981196,0.981351
1,Random Forest,0.971060,0.971118,0.971639
0,Linear SVM,0.873984,0.873147,0.877126


## Model Selection and Analysis Summary

---

### 1. Model Selection
All of the models used some sort of automated feature selection in order to reduce the chances of overfitting. We opted to look at 3 different models, SVM, Random Forest, and XGBoost. SVM was the only linear model we looked at since this problem leaned towards tree based models working better. SVM required us to standardize our variables which was done by using `StandardScaler` through a pipeline. SVM was selected because the given values from the dataset are continuous and real numbers. Random forest was selected since there was a mix of integer and continuous values. XGBoost was selected since there seemed to be a non linear relationship between the features and our target value.

---

### 2. Evaluation Metrics
The evaluation metrics we selected were f1-score, precision and recall. F1 score was selected since our original dataset was imbalanced and this would give us a better metric of how our model performed. Precision and recall were used since having either false positives or false negatives could be dangerous for this situation. False positives could lead to unnecesary emergency measures that could result in more complications. False negatives could cause problems with a fetus to be unnoticed possibly leading to the death of that fetus.

---

### 3. Results
Based on the evaluation metrics we selected, XGBoost performed the best, closely followed by Random Forest and then SVM, confirming that tree based algorithms worked better than linear ones in this case.
